<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:12pt;font-weight:800;font-family:Georgia,serif;">
Gold Layer · Usine C · Génération & Enrichissement
</span><br>
<span style="font-size:9.5pt;font-family:sans-serif;color:#e8e8e8;">
Lecture depuis Silver. Génération du dataset Usine C.
Problème dominant : instabilité process structurelle.
KPIs, TRS, UCL/LCL, export Parquet.
</span>
</div>

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
00 · Imports & Configuration
</span>
</div>

In [2]:
# ============================================================
# GOLD LAYER — USINE C
# Input  : data/silver/ai4i_clean.parquet
# Output : data/gold/usine_C.parquet
#          data/gold/limites_usine_C.json
# ============================================================

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('gold_C')

ROOT        = Path('..').resolve()
SILVER_FILE = ROOT / 'data' / 'silver' / 'ai4i_clean.parquet'
GOLD_PATH   = ROOT / 'data' / 'gold'
GOLD_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

np.random.seed(256)

LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
)

# ── Paramètres physiques Usine C ─────────────────────────────
# Machines standard, vitesse réduite pour REF03
# N = (P × 60) / (2π × T)
PUISSANCE_W      = 5_800.0
COUPLE_NOMINAL   = 38.0
VITESSE_NOMINALE = round((PUISSANCE_W * 60) / (2 * np.pi * COUPLE_NOMINAL))
TEMPS_CYCLE_NOM  = round(60_000 / VITESSE_NOMINALE, 2)
DUREE_VIE_OUTIL  = 250

TEMP_AIR_BASE    = 21.5
OFFSET_THERMIQUE = 9.5

# Taux de rebut par référence — calibré sur la complexité géométrique
REBUT_PAR_PIECE  = {'REF01': 0.03, 'REF02': 0.05, 'REF03': 0.12, 'REF04': 0.07}

# Facteur équipe — fatigue opérateur en nuit
FACTEUR_EQUIPE   = {'Matin': 1.00, 'Apres-midi': 1.10, 'Nuit': 1.35}

# ISP de base par référence
ISP_PAR_PIECE    = {'REF01': 0.92, 'REF02': 0.85, 'REF03': 0.62, 'REF04': 0.78}
FACTEUR_ISP_EQ   = {'Matin': 1.00, 'Apres-midi': 0.97, 'Nuit': 0.88}

USINE_ID         = 'Usine_C'
MACHINES         = ['M01', 'M02', 'M03', 'M04']
TYPES_PIECES     = ['REF01', 'REF02', 'REF03', 'REF04']
# REF03 dominant — cause structurelle du taux rebut élevé
POIDS_PIECES     = [0.25, 0.25, 0.35, 0.15]
N_SESSIONS       = 6 * 30 * 24 * 2
START_DATE       = '2024-01-01 06:00'

log.info("Configuration Gold Usine C chargée")
print(f"\n  Vitesse nominale   : {VITESSE_NOMINALE} tr/min")
print(f"  Temps cycle nominal: {TEMPS_CYCLE_NOM} min")
print(f"\n  Taux rebut par référence :")
for ref, taux in REBUT_PAR_PIECE.items():
    print(f"    {ref} : {taux*100:.0f}%")
print(f"\n  Facteur équipe :")
for eq, f in FACTEUR_EQUIPE.items():
    print(f"    {eq:<12} : ×{f}")

20:36:39 | INFO | Configuration Gold Usine C chargée



  Vitesse nominale   : 1458 tr/min
  Temps cycle nominal: 41.15 min

  Taux rebut par référence :
    REF01 : 3%
    REF02 : 5%
    REF03 : 12%
    REF04 : 7%

  Facteur équipe :
    Matin        : ×1.0
    Apres-midi   : ×1.1
    Nuit         : ×1.35


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
01 · Lecture Silver & structure temporelle
</span>
</div>

In [3]:
# ============================================================
# LECTURE SILVER & STRUCTURE TEMPORELLE USINE C
# ============================================================

df_silver = pd.read_parquet(SILVER_FILE)
log.info("Silver chargé : %d lignes · %d colonnes", *df_silver.shape)

timestamps       = pd.date_range(start=START_DATE, periods=N_SESSIONS, freq='30min')
df               = pd.DataFrame({'timestamp': timestamps})
df['usine_id']   = USINE_ID
df['machine_id'] = np.random.choice(MACHINES, size=N_SESSIONS)
df['type_piece'] = np.random.choice(TYPES_PIECES, size=N_SESSIONS, p=POIDS_PIECES)

def assigner_equipe(ts):
    h = ts.hour
    if 6 <= h < 14:    return 'Matin'
    elif 14 <= h < 22: return 'Apres-midi'
    else:               return 'Nuit'

df['equipe'] = df['timestamp'].apply(assigner_equipe)

print(f"\n  Sessions générées : {len(df):,}")
print(f"  Période           : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"  Répartition pièces (REF03 dominant) :")
print(f"  {df['type_piece'].value_counts().to_dict()}")
print()
display(df.head(5))

20:36:40 | INFO | Silver chargé : 9985 lignes · 15 colonnes



  Sessions générées : 8,640
  Période           : 2024-01-01 06:00:00 → 2024-06-29 05:30:00
  Répartition pièces (REF03 dominant) :
  {'REF03': 3089, 'REF02': 2156, 'REF01': 2097, 'REF04': 1298}



,timestamp,usine_id,machine_id,type_piece,equipe
0,2024-01-01 06:00:00,Usine_C,M01,REF03,Matin
1,2024-01-01 06:30:00,Usine_C,M03,REF01,Matin
2,2024-01-01 07:00:00,Usine_C,M04,REF02,Matin
3,2024-01-01 07:30:00,Usine_C,M02,REF02,Matin
4,2024-01-01 08:00:00,Usine_C,M01,REF04,Matin


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
02 · Génération des capteurs & ISP
</span>
</div>

In [4]:
# ============================================================
# CAPTEURS USINE C — Températures stables, ISP variable
#
# L'ISP (Indice Stabilité Process) est le signal clé Usine C.
# ISP = base_piece × facteur_équipe + bruit
# REF03 + Nuit = combinaison la plus critique
# ============================================================

df['temp_air_c'] = np.round(
    np.random.normal(TEMP_AIR_BASE, 1.2, N_SESSIONS), 2)
df['temp_process_c'] = np.round(
    df['temp_air_c'] + OFFSET_THERMIQUE
    + np.random.normal(0, 0.6, N_SESSIONS), 2)
df['temp_process_delta'] = np.round(
    df['temp_process_c'] - df['temp_air_c'], 2)

# Vitesse adaptée par référence — REF03 nécessite vitesse réduite
vitesse_par_piece = {
    'REF01': (VITESSE_NOMINALE,      50),
    'REF02': (VITESSE_NOMINALE - 20, 55),
    'REF03': (VITESSE_NOMINALE - 80, 60),
    'REF04': (VITESSE_NOMINALE - 40, 55)
}

vitesses, couples = [], []
for piece in df['type_piece']:
    mu_v, sigma_v = vitesse_par_piece[piece]
    vitesses.append(int(np.clip(np.random.normal(mu_v, sigma_v), 900, 1800)))
    couples.append(round(np.clip(np.random.normal(COUPLE_NOMINAL, 5), 20, 60), 2))

df['vitesse_rotation_rpm'] = vitesses
df['couple_nm']            = couples

# Usure outil — remplacement à 82%
increments = {'REF01':(2.0,3.0),'REF02':(2.5,3.5),
              'REF03':(4.0,5.5),'REF04':(2.5,4.0)}

usure_actuelle = 0.0
usure_list, remplacement_list = [], []

for i in range(N_SESSIONS):
    lo, hi = increments[df.loc[i,'type_piece']]
    usure_actuelle += np.random.uniform(lo, hi)
    if usure_actuelle >= DUREE_VIE_OUTIL * 0.82:
        remplacement_list.append(1)
        usure_actuelle = 0.0
    else:
        remplacement_list.append(0)
    usure_list.append(round(usure_actuelle, 1))

df['usure_outil_min']    = usure_list
df['remplacement_outil'] = remplacement_list
df['usure_cumulee_pct']  = np.round(
    (df['usure_outil_min'] / DUREE_VIE_OUTIL * 100).clip(0,100), 2)

# ISP — signal clé Usine C
isp_values = []
for _, row in df.iterrows():
    base    = ISP_PAR_PIECE[row['type_piece']]
    facteur = FACTEUR_ISP_EQ[row['equipe']]
    isp     = base * facteur + np.random.normal(0, 0.04)
    isp_values.append(round(np.clip(isp, 0, 1), 4))

df['indice_stabilite_process'] = isp_values

print(f"  Delta thermique moy : {df['temp_process_delta'].mean():.2f} K  ← stable")
print(f"  Vitesse moyenne     : {df['vitesse_rotation_rpm'].mean():.0f} tr/min")
print(f"  ISP moyen global    : {df['indice_stabilite_process'].mean():.3f}")
print(f"  ISP moyen REF03     : {df[df['type_piece']=='REF03']['indice_stabilite_process'].mean():.3f}")
print(f"  ISP moyen Nuit      : {df[df['equipe']=='Nuit']['indice_stabilite_process'].mean():.3f}")

  Delta thermique moy : 9.50 K  ← stable
  Vitesse moyenne     : 1417 tr/min
  ISP moyen global    : 0.736
  ISP moyen REF03     : 0.589
  ISP moyen Nuit      : 0.680


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
03 · Pannes & maintenance
</span>
</div>

In [5]:
# ============================================================
# PANNES USINE C — Faibles mécaniquement
# Panne process déclenchée quand ISP < 0.55
# Le problème dominant est le rebut, pas les pannes
# ============================================================

prob_usure    = (df['usure_cumulee_pct'] / 100) ** 2 * 0.06
panne_usure   = ((np.random.random(N_SESSIONS) < prob_usure) &
                 (df['usure_cumulee_pct'] > 70)).astype(int)

prob_process  = np.where(df['indice_stabilite_process'] < 0.55, 0.08, 0.01)
panne_process = (np.random.random(N_SESSIONS) < prob_process).astype(int)

panne_surge   = (np.random.random(N_SESSIONS) < 0.002).astype(int)
panne_alea    = (np.random.random(N_SESSIONS) < 0.001).astype(int)

panne_machine = ((panne_usure==1)|(panne_process==1)|
                 (panne_surge==1)|(panne_alea==1)).astype(int)

df['panne_machine']     = panne_machine
df['panne_usure_outil'] = panne_usure
df['panne_thermique']   = panne_process
df['panne_puissance']   = 0
df['panne_surcharge']   = panne_surge
df['panne_aleatoire']   = panne_alea

def type_panne(row):
    if row['panne_usure_outil']: return 'Usure outil'
    if row['panne_thermique']:   return 'Process instable'
    if row['panne_surcharge']:   return 'Surcharge'
    if row['panne_aleatoire']:   return 'Aleatoire'
    return 'Aucune'

df['type_panne'] = df.apply(type_panne, axis=1)

durees = []
for _, row in df.iterrows():
    if row['panne_usure_outil']: durees.append(round(np.random.uniform(45,120),1))
    elif row['panne_thermique']: durees.append(round(np.random.uniform(20,60),1))
    elif row['panne_surcharge']: durees.append(round(np.random.uniform(20,60),1))
    elif row['panne_aleatoire']: durees.append(round(np.random.uniform(10,30),1))
    else: durees.append(0.0)

df['duree_arret_min']  = durees
df['nb_pannes_7j']     = df['panne_machine'].rolling(336, min_periods=1).sum().astype(int)
df['intervention_maintenance'] = (
    (df['panne_machine']==1)|(df['nb_pannes_7j']>5)).astype(int)

print(f"  Taux panne global    : {df['panne_machine'].mean()*100:.2f}%  ← faible")
print(f"  Pannes usure outil   : {df['panne_usure_outil'].sum()}")
print(f"  Pannes process       : {df['panne_thermique'].sum()}")
print(f"  Durée arrêt moy      : {df[df['duree_arret_min']>0]['duree_arret_min'].mean():.1f} min")

  Taux panne global    : 2.49%  ← faible
  Pannes usure outil   : 46
  Pannes process       : 137
  Durée arrêt moy      : 47.8 min


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
04 · KPIs production — rebut structurel
</span>
</div>

In [6]:
# ============================================================
# KPIs USINE C — Rebut structurel
#
# τ = τ_piece × f_equipe × (2 - ISP) + bruit
#
# (2 - ISP) amplifie le rebut quand le process est instable :
#   ISP = 1.0 → facteur = 1.0 → pas d'amplification
#   ISP = 0.6 → facteur = 1.4 → rebut +40%
#   ISP = 0.4 → facteur = 1.6 → rebut +60%
# ============================================================

df['temps_cycle_min'] = np.round(
    (TEMPS_CYCLE_NOM * (1 + 0.001 * df['usure_cumulee_pct'])
     + np.random.normal(0, 2.0, N_SESSIONS)).clip(lower=20), 2)

df['derive_cycle_pct'] = np.round(
    (df['temps_cycle_min'] - TEMPS_CYCLE_NOM) / TEMPS_CYCLE_NOM * 100, 2)

df['pieces_produites'] = (30 / df['temps_cycle_min']).astype(int).clip(lower=1)
df.loc[df['panne_machine']==1, 'pieces_produites'] = 0

taux_rebut_values = []
for _, row in df.iterrows():
    base    = REBUT_PAR_PIECE[row['type_piece']]
    facteur = FACTEUR_EQUIPE[row['equipe']]
    isp     = row['indice_stabilite_process']
    taux    = base * facteur * (2 - isp) + np.random.normal(0, 0.008)
    taux_rebut_values.append(round(np.clip(taux, 0, 1), 4))

df['taux_rebut']      = taux_rebut_values
df['pieces_rebutees'] = (df['pieces_produites'] * df['taux_rebut']).astype(int)

df['consommation_kwh'] = np.round(
    ((df['vitesse_rotation_rpm'] * df['couple_nm']) / 9550 * 0.5
     + np.random.normal(0, 0.35, N_SESSIONS)).clip(lower=0), 2)

mask_appro = (df['timestamp'] >= '2024-03-05') & (df['timestamp'] <= '2024-03-10')
df['rupture_appro'] = mask_appro.astype(int)
df.loc[mask_appro, 'pieces_produites'] = 0

print(f"  Temps cycle moyen       : {df['temps_cycle_min'].mean():.2f} min")
print(f"  Taux rebut moyen        : {df['taux_rebut'].mean()*100:.2f}%  ← structurellement élevé")
print(f"  Taux rebut REF03        : {df[df['type_piece']=='REF03']['taux_rebut'].mean()*100:.2f}%")
print(f"  Taux rebut Nuit         : {df[df['equipe']=='Nuit']['taux_rebut'].mean()*100:.2f}%")
print(f"  Taux rebut REF03 + Nuit : {df[(df['type_piece']=='REF03')&(df['equipe']=='Nuit')]['taux_rebut'].mean()*100:.2f}%")
print(f"  Pièces produites        : {df['pieces_produites'].sum():,}")

  Temps cycle moyen       : 42.82 min
  Taux rebut moyen        : 11.16%  ← structurellement élevé
  Taux rebut REF03        : 19.56%
  Taux rebut Nuit         : 13.70%
  Taux rebut REF03 + Nuit : 23.50%
  Pièces produites        : 8,193


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
05 · TRS & Limites de contrôle SPC
</span>
</div>

In [7]:
# ============================================================
# TRS & SPC — USINE C
# TRS pénalisé par la qualité, pas la disponibilité
# ============================================================

df['disponibilite'] = (df['panne_machine']==0).astype(float)
df['performance']   = np.round((df['vitesse_rotation_rpm']/VITESSE_NOMINALE).clip(upper=1.0), 4)
df['qualite']       = np.round(1 - df['taux_rebut'], 4)
df['TRS']           = np.round(df['disponibilite']*df['performance']*df['qualite'], 4)

baseline = df[(df['timestamp'] < '2024-02-01') & (df['panne_machine']==0)]

def calcul_limites(serie, nom, n_sigma=3):
    mu, sigma = serie.mean(), serie.std()
    return {
        'kpi'    : nom,
        'moyenne': round(mu, 4),
        'sigma'  : round(sigma, 4),
        'ucl'    : round(mu + n_sigma*sigma, 4),
        'lcl'    : round(max(mu - n_sigma*sigma, 0), 4),
        'uwl'    : round(mu + 2*sigma, 4),
        'lwl'    : round(max(mu - 2*sigma, 0), 4),
    }

kpis = {
    'TRS'                : baseline['TRS'],
    'taux_rebut'         : baseline['taux_rebut'],
    'temps_cycle_min'    : baseline['temps_cycle_min'],
    'temp_process_delta' : baseline['temp_process_delta'],
    'consommation_kwh'   : baseline['consommation_kwh'],
    'indice_stabilite'   : baseline['indice_stabilite_process'],
}

limites = {nom: calcul_limites(serie, nom) for nom, serie in kpis.items()}
df_lim  = pd.DataFrame(limites.values()).set_index('kpi')

print(f"  TRS moyen global  : {df['TRS'].mean()*100:.1f}%")
print(f"  Disponibilité moy : {df['disponibilite'].mean()*100:.1f}%  ← bonne")
print(f"  Performance moy   : {df['performance'].mean()*100:.1f}%  ← bonne")
print(f"  Qualité moy       : {df['qualite'].mean()*100:.1f}%  ← problème ici")
print(f"  TRS moyen REF03   : {df[df['type_piece']=='REF03']['TRS'].mean()*100:.1f}%")
print()
display(df_lim)

  TRS moyen global  : 83.8%
  Disponibilité moy : 97.5%  ← bonne
  Performance moy   : 96.5%  ← bonne
  Qualité moy       : 88.8%  ← problème ici
  TRS moyen REF03   : 73.1%



,moyenne,sigma,ucl,lcl,uwl,lwl
kpi,,,,,,
TRS,0.86,0.09,1.12,0.59,1.03,0.68
taux_rebut,0.11,0.07,0.33,0.00,0.26,0.00
temps_cycle_min,42.77,2.17,49.27,36.27,47.10,38.44
temp_process_delta,9.51,0.60,11.30,7.71,10.70,8.31
consommation_kwh,2.82,0.53,4.40,1.23,3.87,1.76
indice_stabilite,0.73,0.13,1.13,0.34,1.00,0.47


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
06 · Visualisations Gold — dérives Usine C
</span>
</div>

In [8]:
# ── Plot 1 : Taux de rebut par type de pièce et équipe ───────
rebut_piece = df.groupby('type_piece')['taux_rebut'].mean().reset_index()
rebut_equipe = df.groupby('equipe')['taux_rebut'].mean().reset_index()

fig1 = make_subplots(rows=1, cols=2,
    subplot_titles=['Taux rebut moyen par référence',
                    'Taux rebut moyen par équipe'],
    horizontal_spacing=0.12)

couleurs_piece = {'REF01':'#10b981','REF02':'#6366f1',
                  'REF03':'#ef4444','REF04':'#f97316'}

for _, row in rebut_piece.iterrows():
    fig1.add_trace(go.Bar(
        x=[row['type_piece']], y=[row['taux_rebut']],
        name=row['type_piece'],
        marker_color=couleurs_piece[row['type_piece']],
        text=[f"{row['taux_rebut']*100:.1f}%"],
        textposition='outside', textfont=dict(size=10),
        hovertemplate=f"<b>{row['type_piece']}</b><br>Rebut : {row['taux_rebut']*100:.2f}%<extra></extra>",
        showlegend=False
    ), row=1, col=1)

couleurs_eq = {'Matin':'#10b981','Apres-midi':'#6366f1','Nuit':'#ef4444'}
for _, row in rebut_equipe.iterrows():
    fig1.add_trace(go.Bar(
        x=[row['equipe']], y=[row['taux_rebut']],
        name=row['equipe'],
        marker_color=couleurs_eq.get(row['equipe'], '#6b7280'),
        text=[f"{row['taux_rebut']*100:.1f}%"],
        textposition='outside', textfont=dict(size=10),
        hovertemplate=f"<b>{row['equipe']}</b><br>Rebut : {row['taux_rebut']*100:.2f}%<extra></extra>",
        showlegend=False
    ), row=1, col=2)

fig1.update_layout(
    **LAYOUT,
    title=dict(text='Usine C · Taux de rebut : Référence vs Équipe',
               font=dict(size=13, color='#111827'), x=0.01),
    height=400, margin=dict(l=50,r=20,t=60,b=40)
)
fig1.update_xaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=10, color='#9ca3af'))
fig1.update_yaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=10, color='#9ca3af'))
for ann in fig1.layout.annotations:
    ann.font.update(size=10, color='#6b7280')
fig1.show()

In [11]:
# ── Plot 2 : TRS Usine C — pénalisé par la qualité ───────────
sample = df.iloc[::3].copy()
df['TRS_7j'] = df['TRS'].rolling(336, min_periods=1).mean()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['TRS'],
    name='TRS session', line=dict(color='#bbf7d0', width=0.8), opacity=0.6,
    hovertemplate='%{x|%d %b %H:%M}<br>TRS : %{y:.3f}<extra></extra>'
))
fig2.add_trace(go.Scatter(
    x=df['timestamp'], y=df['TRS_7j'],
    name='Moyenne mobile 7j', line=dict(color='#10b981', width=2.5),
    hovertemplate='%{x|%d %b}<br>TRS moy 7j : %{y:.3f}<extra></extra>'
))
fig2.add_hline(y=0.85, line_dash='dot', line_color='#ef4444', line_width=1.2,
               annotation_text='Seuil acceptable 85%',
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='bottom right')
fig2.add_hline(y=limites['TRS']['lcl'], line_dash='dot', line_color='#f97316',
               line_width=1,
               annotation_text=f"LCL = {limites['TRS']['lcl']:.3f}",
               annotation_font_size=9, annotation_font_color='#f97316',
               annotation_position='top right')

fig2.update_layout(
    **LAYOUT,
    title=dict(text='Usine C · TRS sur 6 mois : pénalisé par la composante Qualité',
               font=dict(size=13, color='#111827'), x=0.01),
    xaxis=dict(gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), tickformat='%d %b'),
    yaxis=dict(title='TRS', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), range=[0,1.05]),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    height=380, margin=dict(l=50,r=20,t=60,b=40),
    hovermode='x unified'
)
fig2.show()

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
07 · Export Gold
</span>
</div>

In [10]:
# ============================================================
# EXPORT GOLD — USINE C
# ============================================================

def json_serial(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    raise TypeError(f"Type {type(obj)} non sérialisable")

gold_file = GOLD_PATH / 'usine_C.parquet'
lim_file  = GOLD_PATH / 'limites_usine_C.json'

df.to_parquet(gold_file, index=False)

with open(lim_file, 'w') as f:
    json.dump(limites, f, indent=2, default=json_serial)

log.info("Gold Usine C exporté : %s", gold_file)

print(f"\n{'='*55}")
print(f"  GOLD OUTPUTS — USINE C")
print(f"{'='*55}")
print(f"  ✓ {gold_file}")
print(f"  ✓ {lim_file}")
print(f"  Shape             : {df.shape[0]:,} lignes · {df.shape[1]} colonnes")
print(f"  TRS moyen         : {df['TRS'].mean()*100:.1f}%")
print(f"  Disponibilité moy : {df['disponibilite'].mean()*100:.1f}%")
print(f"  Performance moy   : {df['performance'].mean()*100:.1f}%")
print(f"  Qualité moy       : {df['qualite'].mean()*100:.1f}%")
print(f"  Taux rebut moyen  : {df['taux_rebut'].mean()*100:.2f}%")
print(f"  Taux rebut REF03  : {df[df['type_piece']=='REF03']['taux_rebut'].mean()*100:.2f}%")
print(f"{'='*55}")

20:36:44 | INFO | Gold Usine C exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_C.parquet



  GOLD OUTPUTS — USINE C
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_C.parquet
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\limites_usine_C.json
  Shape             : 8,640 lignes · 36 colonnes
  TRS moyen         : 83.8%
  Disponibilité moy : 97.5%
  Performance moy   : 96.5%
  Qualité moy       : 88.8%
  Taux rebut moyen  : 11.16%
  Taux rebut REF03  : 19.56%
